# MLOps Pipelines

Hands-on examples for Module 10, covering:

1. GitHub Actions CI/CD setup
2. Automated model packaging
3. Continuous training pipeline
4. Quality gates and evaluation
5. Monitoring and alerting

## 1. GitHub Actions CI/CD Setup

In [ ]:
# Create GitHub Actions workflow for training
train_workflow = '''
name: Train Model

on:
  push:
    branches: [main]
    paths: ['data/**', 'configs/**']
  workflow_dispatch:  # Manual trigger

jobs:
  train:
    runs-on: ubuntu-latest
    
    steps:
    - uses: actions/checkout@v4
    
    - name: Setup Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.10'
    
    - name: Install dependencies
      run: pip install -r requirements.txt
    
    - name: Download dataset
      run: |
        huggingface-cli download your-org/dataset --local-dir ./data
    
    - name: Train model
      run: python train.py
      env:
        HF_TOKEN: ${{{{ secrets.HF_TOKEN }}}}
        WANDB_API_KEY: ${{{{ secrets.WANDB_API_KEY }}}}
    
    - name: Upload model to Hugging Face
      run: |
        huggingface-cli upload your-org/your-model ./output --revision ${{{{ github.sha }}}}
    
    - name: Notify on success
      if: success()
      uses: slackapi/slack-github-action@v1
      with:
        payload: |
          {"text": "Model training completed successfully!"}
'''

# Save workflow
import os
os.makedirs(".github/workflows", exist_ok=True)

with open(".github/workflows/train-model.yml", "w") as f:
    f.write(train_workflow)

print("GitHub Actions workflow saved to .github/workflows/train-model.yml")

In [ ]:
# Create evaluation workflow
eval_workflow = '''
name: Evaluate Model

on:
  push:
    paths: ['output/**', 'evals/**']

jobs:
  evaluate:
    runs-on: gpu
    
    steps:
    - uses: actions/checkout@v4
    
    - name: Setup Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.10'
    
    - name: Install dependencies
      run: pip install -r requirements.txt lm-eval
    
    - name: Run MMLU
      run: |
        lm_eval --model hf \\
          --model_args pretrained=./output \\
          --tasks mmlu \\
          --batch_size 4 \\
          --output_path ./eval-results/mmlu
    
    - name: Run TruthfulQA
      run: |
        lm_eval --model hf \\
          --model_args pretrained=./output \\
          --tasks truthfulqa \\
          --batch_size 4 \\
          --output_path ./eval-results/truthfulqa
    
    - name: Check quality gate
      run: python check_quality_gate.py ./eval-results
'''

with open(".github/workflows/evaluate-model.yml", "w") as f:
    f.write(eval_workflow)

print("Evaluation workflow saved to .github/workflows/evaluate-model.yml")

## 2. Automated Model Packaging

In [ ]:
# Generate model card
def generate_model_card(model_name, training_config, eval_results):
    """Generate a model card in markdown format."""
    
    card = f"""---
language:
- en
tags:
- llm
- fine-tuned
- text-generation
license: apache-2.0
model-index:
- name: {model_name}
  results: []
---

# {model_name}

A fine-tuned language model based on Llama-3.2-3B.

## Model Details

- **Model type**: Llama-3.2-3B
- **Fine-tuning method**: LoRA (QLoRA)
- **Training data**: Custom instruction dataset
- **License**: Apache 2.0

## Training Configuration

| Parameter | Value |
|-----------|-------|
| Learning Rate | {training_config.get('lr', '2e-5')} |
| Epochs | {training_config.get('epochs', '3')} |
| Batch Size | {training_config.get('batch_size', '16')} |
| LoRA Rank | {training_config.get('lora_r', '16')} |

## Evaluation Results

| Benchmark | Score |
|-----------|-------|
| MMLU | {eval_results.get('mmlu', 'N/A')}% |
| TruthfulQA | {eval_results.get('truthfulqa', 'N/A')}% |
| HellaSwag | {eval_results.get('hellaswag', 'N/A')}% |

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{model_name}")
tokenizer = AutoTokenizer.from_pretrained("{model_name}")

inputs = tokenizer("Hello, how can I help you?", return_tensors="pt")
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0]))
```

## Limitations

- May generate incorrect information for specialized domains
- Training data cutoff: 2024
- English only

## Citation

```bibtex
@model{{{model_name.replace(' ', '_')}},
  title = {{"{model_name}"}},
  year = {{2024}},
}}
```
"""
    return card

# Generate example model card
model_card = generate_model_card(
    model_name="my-org/instruction-model-v1",
    training_config={"lr": "2e-4", "epochs": "3", "batch_size": "16", "lora_r": "16"},
    eval_results={"mmlu": "45.2", "truthfulqa": "38.5", "hellaswag": "58.3"},
)

# Save model card
with open("MODEL_CARD.md", "w") as f:
    f.write(model_card)

print("Model card saved to MODEL_CARD.md")
print("\nPreview:")
print("=" * 60)
print(model_card[:500] + "...")

In [ ]:
# Create Dockerfile for serving
dockerfile = '''
FROM nvidia/cuda:12.0-runtime-ubuntu22.04

WORKDIR /app

# Install Python
RUN apt-get update && apt-get install -y \\
    python3.10 \\
    python3-pip \\
    && rm -rf /var/lib/apt/lists/*

# Install dependencies
COPY requirements.txt .
RUN pip3 install -r requirements.txt

# Copy model
COPY ./output /app/model
COPY server.py .

# Environment variables
ENV MODEL_PATH=/app/model
ENV PORT=8000

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Run server
CMD ["python3", "server.py"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

print("Dockerfile created")

# Create docker-compose.yml
docker_compose = '''
version: '3.8'

services:
  llm-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/model
      - MAX_TOKENS=512
    volumes:
      - ./output:/app/model:ro
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
'''

with open("docker-compose.yml", "w") as f:
    f.write(docker_compose)

print("docker-compose.yml created")

## 3. Continuous Training Pipeline

In [ ]:
import hashlib
from pathlib import Path
from datetime import datetime
import json

class ContinuousTrainingPipeline:
    """Simulated continuous training pipeline."""
    
    def __init__(self, data_dir: str = "./data", model_dir: str = "./output"):
        self.data_dir = Path(data_dir)
        self.model_dir = Path(model_dir)
        self.checkpoint_file = self.model_dir / "data_checkpoint.json"
    
    def compute_data_hash(self) -> str:
        """Compute hash of all data files."""
        hasher = hashlib.md5()
        
        data_files = sorted(self.data_dir.glob("*.jsonl")) if self.data_dir.exists() else []
        
        for data_file in data_files:
            with open(data_file, 'rb') as f:
                hasher.update(f.read())
        
        return hasher.hexdigest() if data_files else "no_data"
    
    def check_new_data(self) -> bool:
        """Check if new data is available."""
        current_hash = self.compute_data_hash()
        
        if not self.checkpoint_file.exists():
            print(f"No checkpoint found. Current hash: {current_hash}")
            return True
        
        with open(self.checkpoint_file) as f:
            last_hash = json.load(f).get("data_hash", "")
        
        is_new = current_hash != last_hash
        print(f"Last hash: {last_hash}")
        print(f"Current hash: {current_hash}")
        print(f"New data: {is_new}")
        
        return is_new
    
    def save_checkpoint(self, metrics: dict):
        """Save training checkpoint."""
        self.model_dir.mkdir(parents=True, exist_ok=True)
        
        checkpoint = {
            "timestamp": datetime.now().isoformat(),
            "data_hash": self.compute_data_hash(),
            "metrics": metrics,
        }
        
        with open(self.checkpoint_file, "w") as f:
            json.dump(checkpoint, f, indent=2)
        
        print(f"Checkpoint saved: {checkpoint}")
    
    def run(self):
        """Run the continuous training pipeline."""
        print("=" * 60)
        print("Continuous Training Pipeline")
        print("=" * 60)
        
        if self.check_new_data():
            print("\n✓ New data detected! Training would start...")
            # In production: subprocess.run(["python", "train.py"])
        else:
            print("\nNo new data. Skipping training.")

# Demo
pipeline = ContinuousTrainingPipeline()
pipeline.run()

## 4. Quality Gates

In [ ]:
# Quality gate configuration
QUALITY_GATES = {
    "mmlu": {"min": 40.0, "target": 50.0},
    "truthfulqa": {"min": 30.0, "target": 40.0},
    "hellaswag": {"min": 50.0, "target": 60.0},
}

def check_quality_gate(eval_results: dict) -> tuple:
    """Check if model passes quality gates."""
    results = []
    all_passed = True
    
    print("\nQuality Gate Results:")
    print("=" * 60)
    
    for metric, thresholds in QUALITY_GATES.items():
        actual = eval_results.get(metric, 0)
        
        if actual < thresholds["min"]:
            status = "❌ FAIL"
            all_passed = False
        elif actual < thresholds["target"]:
            status = "⚠️  BELOW TARGET"
        else:
            status = "✓ PASS"
        
        results.append({
            "metric": metric,
            "actual": actual,
            "min": thresholds["min"],
            "target": thresholds["target"],
            "status": status,
        })
        
        print(f"{metric:15} | Actual: {actual:5.1f} | Min: {thresholds['min']:5.1f} | Target: {thresholds['target']:5.1f} | {status}")
    
    print("=" * 60)
    print(f"Overall: {'✓ ALL GATES PASSED' if all_passed else '❌ GATES FAILED'}")
    
    return all_passed, results

# Test with sample results
sample_results = {
    "mmlu": 45.2,
    "truthfulqa": 35.0,
    "hellaswag": 58.3,
}

passed, details = check_quality_gate(sample_results)

In [ ]:
# Visualize quality gate results
import matplotlib.pyplot as plt
import numpy as np

def plot_quality_gates(results: list):
    """Plot quality gate results."""
    metrics = [r["metric"] for r in results]
    actual = [r["actual"] for r in results]
    mins = [r["min"] for r in results]
    targets = [r["target"] for r in results]
    
    x = np.arange(len(metrics))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    bars1 = ax.bar(x - width, mins, width, label='Minimum', color='#d73a49')
    bars2 = ax.bar(x, targets, width, label='Target', color='#f9d371')
    bars3 = ax.bar(x + width, actual, width, label='Actual', color='#238636')
    
    ax.set_ylabel('Score (%)')
    ax.set_title('Quality Gate Results')
    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 100)
    
    # Add value labels
    for bar in bars3:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

plot_quality_gates(details)

## 5. Monitoring and Alerting

In [ ]:
# Simulated monitoring dashboard data
import random
from datetime import datetime, timedelta

def generate_metrics_data(hours: int = 24) -> list:
    """Generate simulated metrics data."""
    data = []
    base_time = datetime.now() - timedelta(hours=hours)
    
    for i in range(hours):
        timestamp = base_time + timedelta(hours=i)
        
        # Simulate metrics with some anomalies
        latency = random.gauss(150, 30)  # ms
        if i == hours // 2:  # Simulate spike
            latency *= 3
        
        throughput = random.gauss(50, 10)  # tokens/sec
        error_rate = random.gauss(0.5, 0.2)  # %
        
        data.append({
            "timestamp": timestamp.isoformat(),
            "latency_p95_ms": max(50, latency),
            "throughput_tok_s": max(10, throughput),
            "error_rate_pct": max(0, error_rate),
        })
    
    return data

# Generate and plot metrics
metrics_data = generate_metrics_data(24)

timestamps = [d["timestamp"] for d in metrics_data]
latencies = [d["latency_p95_ms"] for d in metrics_data]
throughputs = [d["throughput_tok_s"] for d in metrics_data]
errors = [d["error_rate_pct"] for d in metrics_data]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Latency
axes[0].plot(timestamps, latencies, 'b-', linewidth=2, label='p95 Latency')
axes[0].axhline(y=200, color='r', linestyle='--', label='Threshold (200ms)')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('P95 Latency Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Throughput
axes[1].plot(timestamps, throughputs, 'g-', linewidth=2, label='Throughput')
axes[1].axhline(y=30, color='r', linestyle='--', label='Min Threshold (30 tok/s)')
axes[1].set_ylabel('Tokens/sec')
axes[1].set_title('Throughput Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

# Error rate
axes[2].plot(timestamps, errors, 'r-', linewidth=2, label='Error Rate')
axes[2].axhline(y=1.0, color='orange', linestyle='--', label='Warning (1%)')
axes[2].axhline(y=2.0, color='red', linestyle='--', label='Critical (2%)')
axes[2].set_ylabel('Error Rate (%)')
axes[2].set_title('Error Rate Over Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Alert manager simulation
class AlertManager:
    """Simulated alert manager."""
    
    def __init__(self):
        self.alerts = []
    
    def check_and_alert(self, metrics: dict):
        """Check metrics and generate alerts."""
        alerts = []
        
        # Latency check
        if metrics.get("latency_p95_ms", 0) > 200:
            alerts.append({
                "severity": "WARNING",
                "metric": "latency_p95_ms",
                "value": metrics["latency_p95_ms"],
                "threshold": 200,
                "message": f"High latency: {metrics['latency_p95_ms']:.0f}ms > 200ms",
            })
        
        # Error rate check
        if metrics.get("error_rate_pct", 0) > 1.0:
            severity = "CRITICAL" if metrics["error_rate_pct"] > 2.0 else "WARNING"
            alerts.append({
                "severity": severity,
                "metric": "error_rate_pct",
                "value": metrics["error_rate_pct"],
                "threshold": 1.0 if severity == "WARNING" else 2.0,
                "message": f"{severity}: Error rate {metrics['error_rate_pct']:.1f}%",
            })
        
        # Throughput check
        if metrics.get("throughput_tok_s", 100) < 30:
            alerts.append({
                "severity": "WARNING",
                "metric": "throughput_tok_s",
                "value": metrics["throughput_tok_s"],
                "threshold": 30,
                "message": f"Low throughput: {metrics['throughput_tok_s']:.1f} tok/s < 30 tok/s",
            })
        
        self.alerts.extend(alerts)
        return alerts
    
    def print_alerts(self):
        """Print all alerts."""
        if not self.alerts:
            print("No alerts.")
            return
        
        print("\n" + "=" * 60)
        print("ALERTS")
        print("=" * 60)
        for alert in self.alerts:
            print(f"[{alert['severity']}] {alert['message']}")
        print("=" * 60)

# Test alert manager
alert_manager = AlertManager()

print("Checking metrics for alerts...")
for i, data in enumerate(metrics_data):
    alerts = alert_manager.check_and_alert(data)
    if alerts and i % 6 == 0:  # Print some alerts
        print(f"\nHour {i}: {len(alerts)} alert(s)")
        for alert in alerts:
            print(f"  - {alert['message']}")

print(f"\nTotal alerts generated: {len(alert_manager.alerts)}")

## Summary

This notebook covered:

1. **GitHub Actions CI/CD**: Automated training and evaluation workflows
2. **Model Packaging**: Model cards, Dockerfiles, containerization
3. **Continuous Training**: Data hash checking, checkpoint management
4. **Quality Gates**: Pass/fail thresholds with visualization
5. **Monitoring**: Latency, throughput, error tracking with alerting

### Key Takeaways:

- Automate training on data/code changes with CI/CD
- Use model cards for documentation and reproducibility
- Quality gates prevent degraded models from deploying
- Monitor latency, throughput, and errors in production
- Set up alerts for anomalies and threshold violations